In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install bitsandbytes trl

In [ ]:
import bitsandbytes as bnb
import os, json, random, warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
 
import torch
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from huggingface_hub import login


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_Token2")
login(token=hf_token)

In [ ]:
FFT_PATH = "/kaggle/input/datasets/bhavyranka/fd-llm-crwu-dataset/cwru_fft_dataset.json"

In [ ]:
with open(FFT_PATH) as f:
    fft_data = json.load(f)
 
print(f"Total samples      : {len(fft_data)}")
print(f"Label distribution : {Counter(d['output'] for d in fft_data)}")
print(f"Sample instruction : {fft_data[0]['instruction'][:100]}...")
print(f"Sample output      : {fft_data[0]['output']}")

In [ ]:
random.seed(42)
np.random.seed(42)
 
LOAD_TO_RPM = {0: 1797, 1: 1772, 2: 1750, 3: 1730}
 
def get_subset(data, load):
    key = f"{load} hp, {LOAD_TO_RPM[load]} rpm"
    return [d for d in data if key in d['instruction']]
 
train_t1, test_t1 = train_test_split(
    fft_data, test_size=0.1, random_state=42,
    stratify=[d['output'] for d in fft_data]
)
 
subset_0hp = get_subset(fft_data, 0)
subset_1hp = get_subset(fft_data, 1)
subset_2hp = get_subset(fft_data, 2)
subset_3hp = get_subset(fft_data, 3)
 
train_t2, test_t2_0hp = train_test_split(
    subset_0hp, test_size=0.1, random_state=42,
    stratify=[d['output'] for d in subset_0hp]
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           
    bnb_4bit_use_double_quant=True,      
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token      
tokenizer.padding_side = "right"                
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
def format_samples(examples):
    texts = []
    for instr, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        messages = [
            {"role": "system",    "content": instr},
            {"role": "user",      "content": inp},
            {"role": "assistant", "content": out},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

def make_hf_dataset(data_list):
    ds = Dataset.from_pandas(pd.DataFrame(data_list))
    return ds.map(format_samples, batched=True)

train_t1, eval_t1 = train_test_split(train_t1, test_size=0.1, random_state=42)
eval_dataset = make_hf_dataset(eval_t1)
train_ds_t1  = make_hf_dataset(train_t1)
train_ds_t2  = make_hf_dataset(train_t2)


In [ ]:
print(eval_dataset.column_names)

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,             
    lora_alpha=16,   
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[  
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "gate_proj", 
        "up_proj", 
        "down_proj"
    ],
)

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

clean_train_dataset = Dataset.from_pandas(train_ds_t1.to_pandas())
clean_eval_dataset = Dataset.from_pandas(eval_dataset.to_pandas())

final_config = SFTConfig(
    output_dir="/kaggle/working",
    dataset_text_field="text",
    max_length=300,              
    packing=True,                   
    # Training Arguments
    
    bf16=True,
    do_eval=False,
    eval_strategy="no",
    learning_rate=5e-04,
    num_train_epochs=2,         
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,           
    lr_scheduler_type="cosine",
    push_to_hub=True,
    hub_model_id="bhavyranka09/llama3-8b-instruct-alpaca-t1",
    report_to="none",
    save_strategy="steps",
    save_steps=100,             
    save_total_limit=2,
    seed=42,
    warmup_steps=10,  
    dataloader_num_workers = 4,
    dataloader_pin_memory = True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=clean_train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=final_config,
)

total_steps = len(trainer.get_train_dataloader()) * final_config.num_train_epochs
print(f"Total training steps: {total_steps}")

torch.cuda.empty_cache()

In [ ]:
trainer.train(
   resume_from_checkpoint="/kaggle/input/datasets/bhavyranka/checkpoint-600"
)
trainer.model.save_pretrained("/kaggle/working/model_checkpoint")
tokenizer.save_pretrained("/kaggle/working/model_checkpoint")
print(":)done")

In [ ]:
# shutil.make_archive('checkpoint_600_backup', 'zip', '/kaggle/working/checkpoint-600')

In [ ]:
model.save_pretrained("kaggle/working")
tokenizer.save_pretrained("kaggle/working")

In [ ]:
find /kaggle/working -mindepth 1 \
! -name "checkpoint-748" \
! -path "/kaggle/working/checkpoint-748/*" \
-exec rm -rf {} +

In [ ]:
from peft import PeftModel

# Load base model fresh
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/checkpoint-748",  
)
model.eval()
print("✅ Fine-tuned model loaded from checkpoint-748")

In [ ]:
sample = test_t1[:2]
prompts = [INFER_TEMPLATE.format(instruction=d["instruction"], input=d["input"]) for d in sample]

inputs = tokenizer(prompts, return_tensors="pt", padding=True, 
                   truncation=True, max_length=800).to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=20, 
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)

input_len = inputs["input_ids"].shape[1]
for i, out in enumerate(outputs):
    decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True)
    print(f"Sample {i+1} raw output: '{decoded}'")
    print(f"True label: {sample[i]['output']}")

In [ ]:
INFER_TEMPLATE_MESSAGES = lambda instr, inp: [
    {"role": "system", "content": instr},
    {"role": "user",   "content": inp},
]

def evaluate_model(model, tokenizer, test_data, split_name="test", batch_size=4):
    tokenizer.padding_side = "left"   
    model.eval()
    preds, trues = [], []

    for i in range(0, len(test_data), batch_size):
        batch = test_data[i : i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                INFER_TEMPLATE_MESSAGES(d["instruction"], d["input"]),
                tokenize=False,
                add_generation_prompt=True   
            )
            for d in batch
        ]
        prompt_lengths = [
            tokenizer(p, return_tensors="pt").input_ids.shape[1]
            for p in prompts
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=800
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for j, (out, plen) in enumerate(zip(outputs, prompt_lengths)):

            decoded = tokenizer.decode(out[plen:], skip_special_tokens=True)
            preds.append(extract_label(decoded))
            trues.append(batch[j]["output"])

        if (i // batch_size) % 20 == 0:
            print(f"  [{i}/{len(test_data)}] done...")


    valid         = [(p, t) for p, t in zip(preds, trues) if p != "UNKNOWN"]
    unknown_count = len(preds) - len(valid)
    vp, vt        = zip(*valid) if valid else ([], [])

    acc  = accuracy_score(vt, vp)
    f1   = f1_score(vt, vp,  average="weighted", labels=LABELS, zero_division=0)
    prec = precision_score(vt, vp, average="weighted", labels=LABELS, zero_division=0)
    rec  = recall_score(vt, vp,  average="weighted", labels=LABELS, zero_division=0)
    cm   = confusion_matrix(vt, vp, labels=LABELS)

    print(f"\n{'='*50}\nResults — {split_name}\n{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  Unknowns  : {unknown_count}/{len(preds)}")

    return {"split": split_name, "accuracy": acc, "f1": f1,
            "precision": prec, "recall": rec, "unknowns": unknown_count}

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   
result_t1 = evaluate_model(model, tokenizer, test_t1, "Task1_combined")